# Reproduce Study 1

Install `requirements.txt`, then choose **Run all**. This notebook uses the same code as `python -m scripts.run`.

The main dataset is `data/regression_data.csv`. The code rebuilds it from the released coded sources, checks the structural measures, and reproduces the tables, empirical figure, regressions, and robustness checks.

Results are saved in `results/`; rebuilt datasets and coding checks are in `data/`. The [data guide](data/README.md) explains the inputs and coding.


## 1. Rebuild everything

The command below checks all payoff calculations, joins the coded outcomes and controls, and constructs the main and stricter datasets. It then fits the five models, the negotiation-count weighted comparison, and the planned checks. Resampling uses 5,000 complete-observation draws with recorded seeds. The final step produces the paper's tables and figures. The run may take a few minutes.

The coding rules and source locations are described in `README.md` and in `data/README.md`. This code checks and processes the released coding; it does not replace reading the papers or deciding which teaching materials they used.


The same run also saves the separate Appendix C tests of the original registration, using the corrected combined-payoff data. Those original tests use their own hypothesis labels and the full original confirmation rules; they are separate from the current experiment-level tests.


In [ ]:
from pathlib import Path
from scripts.run import reproduce

results = reproduce()
data = results.parent / "data"
check_dir = results / "checks"


## 2. Check what went into the analysis

The main dataset retains all included observations. The stricter check removes unresolved assumptions that can change an outcome or predictor. One experiment contributes only the conditions whose separate outcomes and reservation values can be recovered. The eligibility file explains every decision. All 103 included payoff records must have a successful metric replay.

Approval rules are in `data/approval_rules.csv` and the evidence workbook. If no alternative rule is stated, unanimity is the default. The data builder checks each coded rule against the numeric control before combining conditions.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image

display(pd.read_csv(data / "derived/observation_eligibility.csv"))
checks = pd.read_csv(data / "derived/payoff_metric_checks.csv")
display(checks.groupby(["input_type", "recomputed_from_included_input"]).size().rename("Payoff records").reset_index())
assert checks.recomputed_from_included_input.all()
print(json.loads((check_dir / "analysis_summary.json").read_text()))


## 3. Read the results

Table 1 gives descriptive statistics and correlations. Table 2 reports the five main regression models. Table 3 reports the required stability checks and supplementary analyses. Tables C1–C3 give the complete regressions for the stricter and combined-payoff comparisons.

The primary H1 test asks whether search complexity has a negative coefficient with a two-sided p value below .05 in Model 3. The primary H2 comparison asks whether search complexity and controls (Model 2) explain more variance than bargaining-zone size and the same controls (Model 5).

A full-sample result alone is insufficient for confirmation. H1 also requires its 95% interval from 5,000 individual-observation resamples to stay below zero, and its coefficient to stay negative with p < .05 after removing each observation in turn. H2 also requires a resampling interval for the R-squared difference entirely above zero, and a positive difference after every single-observation omission. Each requirement is reported separately in `hypothesis_tests.csv`; `full_confirmation_met` is true only if all pass.

These confirmation checks matter because the practical stopping rule does not fix the final analysis sample size: it depends on the archival evidence we can recover. An interval including zero or a failed omission would mean that the full confirmation criterion is not met, even if the primary result passes.

The weighted check gives observations with more negotiation attempts more weight; it excludes observations without a usable count. The stricter check removes unresolved coding assumptions. The combined-payoff comparison gives one row to each coded payoff record and fits the five models. The separate original-main appendix below applies the original registration's hypothesis labels and confirmation rules to those combined-payoff data.

The five robustness analyses examine strict inclusion, the mix of observations, individual influential observations, negotiation-count weights, and repeat uses of payoff information. The resampling and omission checks also form part of the main hypothesis criteria. The weighted model checks the search-complexity coefficient; the comparisons of Models 2 and 5 assess its explanatory advantage.


In [ ]:
display(Markdown((results / "tables/README.md").read_text(encoding="utf-8")))
display(pd.read_csv(check_dir / "hypothesis_tests.csv"))
display(pd.read_csv(check_dir / "bootstrap_intervals.csv"))
display(pd.read_csv(check_dir / "leave_one_out.csv").groupby(["dataset", "weighted"]).agg(
    checks=("omitted_observation", "size"),
    smallest_coefficient=("search_complexity_b", "min"),
    largest_coefficient=("search_complexity_b", "max"),
    negative_and_significant=("negative_and_significant", "sum")))


## 4. See the figures and find the files

Every table, figure and detailed check is saved in `results/`. The small `regression_data.csv` is ready for analysis; `data/derived/study_inputs.csv` retains its sources and aggregation notes. `data/derived/component_evidence.csv` shows exactly how the source conditions were combined. Please use those evidence files when checking an individual observation.


In [ ]:
for image in sorted((results / "figures").glob("*.png")):
    display(Image(filename=str(image)))
print("Saved results in results/ and rebuilt data in data/.")


## 5. Original registration: main hypothesis tests

[`scripts/original_main_analysis.py`](scripts/original_main_analysis.py) applies the original four hypotheses to the same source-corrected data after combining independent experiment outcomes within each coded payoff record. Some records contain a documented combination of payoff versions. The five regression fits are the same ones shown in Table C2.

For this appendix, H1 concerns the bargaining-zone coefficient in Model 4; H2 concerns search complexity in Model 1; H3 compares the R-squared values of Models 2 and 5; and H4 concerns search complexity in Model 3. The original plan requires each primary test to pass, its 95% interval from 5,000 individual-row resamples to exclude zero in the predicted direction, and every single-row omission to retain support. The table reports each requirement separately before giving the combined decision. These checks select individual coded-payoff rows. No additional analyses from the original supplementary plan are run here.


In [ ]:
display(pd.read_csv(check_dir / "original_main_hypotheses.csv"))
display(pd.read_csv(check_dir / "original_main_bootstrap.csv"))
print(json.loads((check_dir / "original_main_summary.json").read_text(encoding="utf-8"))["confirmation"])
